---
title: "Aggregate Module: Spatial Aggregation to Healthsheds"
---

## aggregate

> This module aggregates the downloaded data into the respective output dataframes.

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

We prototyped the code in this module using a Jupyter notebook. The notebook is available in `notes/prototypes/learning_aggregations_w_michelle_20250328.ipynb`. The code in this module is a cleaned-up version of the code in that notebook. The notebook contains additional comments and explanations of the code, which may be helpful for understanding the code in this module.

The basic process is as follows:

1. Load the netCDF data in memory
2. Statistically aggregate the hourly data to daily data per exposure using resample()
3. Write out the data to tiff
4. Read the tiff data back in
5. Read in the shapefile that defines the healthsheds
6. Spatially aggregate the exposure data to the healthsheds
7. Quality check the aggregations
8. Write out final aggregations to tiff

In [ ]:
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

We're going to write a function that aggregates the data for a single exposure from a file. This file should be the single month data we got from the previous step in the pipeline.

In [ ]:
eg_file = here() / "data/input/nepal_2017_11.nc"

In [0]:
#| echo: false
#| output: asis
show_doc(resample_netcdf)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L34){target="_blank" style="float:right; font-size:smaller"}

### resample_netcdf

>      resample_netcdf (fpath:str, resample:str='1D', agg_func:<built-
>                       infunctioncallable>=<function mean at 0x149a669eccf0>,
>                       time_dim:str='valid_time', **xr_open_kwargs)

*Resample a netCDF file to a specified frequency and aggregation method.

Args:
    fpath (str): Path to the netCDF file.
    resample (str): Resampling frequency (e.g., '1H', '1D').
    agg_func (callable): Aggregation function (e.g., np.mean, np.sum).

Returns:
    xarray.Dataset: Resampled dataset.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| fpath | str |  | Path to the netCDF file. |
| resample | str | 1D | Resampling frequency (e.g., '1H', '1D') |
| agg_func | callable | mean | Aggregation function (e.g., np.mean, np.sum). |
| time_dim | str | valid_time | Name of the time dimension in the dataset. |
| xr_open_kwargs | VAR_KEYWORD |  |  |
| **Returns** | **Dataset** |  | **keywords for python's xarray module** |

In [ ]:
var = 'swvl1'
agg_func = _get_callable(cfg['aggregation']['aggregation'][var]['hourly_to_daily'][0]['function'])

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:

    ds_path = handler.get_dataset("instant")
    resampled_data = resample_netcdf(ds_path, agg_func=agg_func)

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 1
----> 1 with ClimateDataFileHandler(eg_file) as handler:
      3     ds_path = handler.get_dataset("instant")
      4     resampled_data = resample_netcdf(ds_path, agg_func=agg_func)

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/core.py:242, in __enter__(self)
    240 @patch
    241 def __enter__(self:ClimateDataFileHandler):
--> 242     self.prepare()
    243     return self

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/core.py:192, in ClimateDataFileHandler.prepare(self)
    189     return
    191 if not self.original_path.exists():
--> 192     raise FileNotFoundError(f"{self.original_path} does not exist")
    194 # Detect ZIP by magic number
    195 # chatgpt implementation here; this is a common way

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/nepal_2017_11.nc does not exist

I'm going to use a dataclass to represent the tiff data. This will allow us to easily pass around the data and metadata associated with the tiff file. Why? Because I can, lol (I've never used dataclasses and I'm curious about them). ChatGPT thinks this will make the code cleaner and easier to read.

In [0]:
#| echo: false
#| output: asis
show_doc(RasterFile)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L63){target="_blank" style="float:right; font-size:smaller"}

### RasterFile

>      RasterFile (path:str, band:int)

Next, a function to write and read the netCDF to tiff:

In [0]:
#| echo: false
#| output: asis
show_doc(netcdf_to_tiff)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L90){target="_blank" style="float:right; font-size:smaller"}

### netcdf_to_tiff

>      netcdf_to_tiff (ds:xarray.core.dataset.Dataset, band:int, variable:str,
>                      crs:str='EPSG:4326')

*Convert a netCDF file to a GeoTIFF file.

Args:
    fpath (str): Path to the netCDF file.
    output_path (str): Path to save the output GeoTIFF file.
    variable_name (str): Name of the variable to convert.
    time_index (int): Index of the time dimension to extract.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| ds | Dataset |  | The aggregated xarray dataset to convert. |
| band | int |  | The day to rasterise; 1 indexed just like human english |
| variable | str |  | The variable name to convert. |
| crs | str | EPSG:4326 | Coordinate reference system (default is WGS84). |

Now to test it:

In [ ]:
with ClimateDataFileHandler(eg_file) as handler:
    ds_path = handler.get_dataset("instant")
    resampled_nc = resample_netcdf(ds_path)

print(resampled_nc)
resampled_tiff = netcdf_to_tiff(
    ds=resampled_nc,
    band=28,
    variable="swvl1",
    crs="EPSG:4326"
)

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 1
----> 1 with ClimateDataFileHandler(eg_file) as handler:
      2     ds_path = handler.get_dataset("instant")
      3     resampled_nc = resample_netcdf(ds_path)

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/core.py:242, in __enter__(self)
    240 @patch
    241 def __enter__(self:ClimateDataFileHandler):
--> 242     self.prepare()
    243     return self

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/core.py:192, in ClimateDataFileHandler.prepare(self)
    189     return
    191 if not self.original_path.exists():
--> 192     raise FileNotFoundError(f"{self.original_path} does not exist")
    194 # Detect ZIP by magic number
    195 # chatgpt implementation here; this is a common way to check for zip fil

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/nepal_2017_11.nc does not exist

In [ ]:
resampled_tiff.data.shape, resampled_tiff.transform, resampled_tiff.crs, resampled_tiff.bounds

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 resampled_tiff.data.shape, resampled_tiff.transform, resampled_tiff.crs, resampled_tiff.bounds

NameError: name 'resampled_tiff' is not defined


NameError: name 'resampled_tiff' is not defined

Super cool! The tiff file is created and the data is read back in correctly. Now we can move on to the next step, which is to aggregate the data by healthshed.

## Polygon to Raster Cells

This function was initially shared from a previous NSAPH aggregation pipeline [here](https://github.com/NSAPH-Data-Processing/air_pollution__aqdh/blob/2a8109075fe7a8fbf7c435cc34ffa97b63f5e133/utils/faster_zonal_stats.py#L17). To better understand this, here is a ChatGPT explanation of the code:

> This function, [`polygon_to_raster_cells`](https://TinasheMTapera.github.io/era5_sandbox/aggregate.html#polygon_to_raster_cells), is doing a crucial first step in spatial alignment: it determines which raster cells are “touched” by each polygon geometry (e.g., administrative areas, watersheds, etc.).    
Essentially, this function helps figure out which pixels from a raster image fall inside each polygon (like a district, region, or shape). It does this by looking at each polygon one by one, zooming in on just the part of the raster that overlaps with that shape, and marking the pixels that are inside. This is kind of like placing a cookie cutter (the polygon) on a pixelated map (the raster) and seeing which pixels get cut.  
The result is a list where each item tells you the pixel locations that match a specific polygon. You can then use those pixel locations to pull out data from the raster, like temperatures or rainfall, and calculate statistics (like the average) for each shape. This is a key step when you want to summarize raster data within specific regions, like figuring out the average temperature in each county or how much vegetation is in each park.

In [0]:
#| echo: false
#| output: asis
show_doc(polygon_to_raster_cells)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L121){target="_blank" style="float:right; font-size:smaller"}

### polygon_to_raster_cells

>      polygon_to_raster_cells (vectors, raster, nodata=None, affine=None,
>                               all_touched=False, verbose=False, **kwargs)

*Returns an index map for each vector geometry to indices in the raster source.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| vectors |  |  |  |
| raster |  |  |  |
| nodata | NoneType | None |  |
| affine | NoneType | None |  |
| all_touched | bool | False |  |
| verbose | bool | False |  |
| kwargs | VAR_KEYWORD |  |  |
| **Returns** | **dict** |  | **A dictionary mapping vector the ids of geometries to locations (indices) in the raster source.** |

To use this, we must define the polygon and raster data. The polygon data is the healthshed shapefile, and the raster data is the tiff file we created earlier. We can use the [`GoogleDriver`](https://TinasheMTapera.github.io/era5_sandbox/core.html#googledriver) class we defined in `core` to read in the shapefile.

In [ ]:
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = driver.get_drive()
healthsheds = driver.read_healthsheds("Nepal_Healthsheds2024.zip")

In [ ]:
res_poly2cell=polygon_to_raster_cells(
    vectors = healthsheds.geometry.values, # the geometries of the shapefile of the regions
    raster=resampled_tiff.data, # the raster data above
    nodata=resampled_tiff.nodata, # any intersections with no data, may have to be np.nan
    affine=resampled_tiff.transform, # some math thing need to revise
    all_touched=True, 
    verbose=True
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 3
      1 res_poly2cell=polygon_to_raster_cells(
      2     vectors = healthsheds.geometry.values, # the geometries of the shapefile of the regions
----> 3     raster=resampled_tiff.data, # the raster data above
      4     nodata=resampled_tiff.nodata, # any intersections with no data, may have to be np.nan
      5     affine=resampled_tiff.transform, # some math thing need to revise
      6     all_touched=True, 
      7     verbose=True
      8 )

NameError: name 'resampled_tiff' is not defined


NameError: name 'resampled_tiff' is not defined

The data below maps which grid entries fall into each of the regions in the shapefile (e.g. which pixel is in which state)

In [ ]:
res_poly2cell[:5]

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 res_poly2cell[:5]

NameError: name 'res_poly2cell' is not defined


NameError: name 'res_poly2cell' is not defined

Last but not least, we aggregate these data to the healthshed level. We can use the `rasterstats` package to do this.

In [0]:
#| echo: false
#| output: asis
show_doc(aggregate_to_healthsheds)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L195){target="_blank" style="float:right; font-size:smaller"}

### aggregate_to_healthsheds

>      aggregate_to_healthsheds (res_poly2cell:list, raster:__main__.RasterFile,
>                                shapes:geopandas.geodataframe.GeoDataFrame,
>                                names_column:str='fs_uid',
>                                aggregation_func:<built-
>                                infunctioncallable>=<function nanmean at
>                                0x149a6674d270>, aggregation_name:str='mean')

*Aggregate the raster data to the health sheds.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| res_poly2cell | list |  | the result of polygon_to_raster_cells |
| raster | RasterFile |  | the raster data |
| shapes | GeoDataFrame |  | the shapes of the health sheds |
| names_column | str | fs_uid | the unique identifier column name of the health sheds |
| aggregation_func | callable | nanmean | the aggregation function |
| aggregation_name | str | mean | the name of the aggregation function |
| **Returns** | **GeoDataFrame** |  |  |

And now we apply it:

In [ ]:
result = aggregate_to_healthsheds(
    res_poly2cell=res_poly2cell,
    raster=resampled_tiff,
    shapes=healthsheds,
    names_column="fid",
    aggregation_func=np.nanmean,
    aggregation_name="mean_soil_moisture"
)
result.head()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 2
      1 result = aggregate_to_healthsheds(
----> 2     res_poly2cell=res_poly2cell,
      3     raster=resampled_tiff,
      4     shapes=healthsheds,
      5     names_column="fid",
      6     aggregation_func=np.nanmean,
      7     aggregation_name="mean_soil_moisture"
      8 )
      9 result.head()

NameError: name 'res_poly2cell' is not defined


NameError: name 'res_poly2cell' is not defined

And plot for QA:

In [ ]:
result.plot(column="mean_soil_moisture", legend=True)
plt.title("Mean Soil Moisture (m^3 m^-3) by Health Shed Nov 2017 day 1")
plt.show()

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 1
----> 1 result.plot(column="mean_soil_moisture", legend=True)
      2 plt.title("Mean Soil Moisture (m^3 m^-3) by Health Shed Nov 2017 day 1")
      3 plt.show()

NameError: name 'result' is not defined


NameError: name 'result' is not defined

That looks great! The data is aggregated to the healthshed level, and we can see the differences in exposure across the healthsheds. We can also see that the data is not uniform across the healthsheds, which is what we expect.

## Tests and Main

Now we can wrap this up in a main function that will simply take in the input file and generate this output. We can also add some tests to make sure the data is aggregated correctly; tests will run automatically in this notebook.

In [ ]:
import random

In [ ]:
#| eval: false

variables = ["t2m", "d2m"]
years = ["20{:02d}".format(m) for m in range(9, 24)]
months = [str(m) for m in range(1, 13)]
aggregations = [
    ("Mean", np.nanmean),
    ("Max", np.nanmax),
    ("Min", np.nanmin)
]

exposure_variable = random.choice(variables)
year = random.choice(years)
month = random.choice(months)
aggregation_str, agg_func = random.choice(aggregations)
input_file = here() / "data/input/{}_{}.nc".format(year, month)

with initialize(version_base=None, config_path="../conf"):
    cfg = compose(config_name='config.yaml')

driver = GoogleDriver(json_key_path=here() / cfg.GOOGLE_DRIVE_AUTH_JSON.path)
drive = driver.get_drive()
healthsheds = driver.read_healthsheds(cfg.GOOGLE_DRIVE_AUTH_JSON.healthsheds_id)

with ClimateDataFileHandler(input_file) as handler:
    ds_path = handler.get_dataset("instant")
    resampled_nc_file = resample_netcdf(ds_path, agg_func=agg_func)

days = len(resampled_nc_file.valid_time.values)
day = random.choice(range(1, days + 1))

resampled_tiff = netcdf_to_tiff(
    ds=resampled_nc_file,
    band=day, # the day we're aggregating
    variable=exposure_variable,
    crs="EPSG:4326"
)

res_poly2cell=polygon_to_raster_cells(
    vectors = healthsheds.geometry.values, # the geometries of the shapefile of the regions
    raster=resampled_tiff.data, # the raster data above
    nodata=resampled_tiff.nodata, # any intersections with no data, may have to be np.nan
    affine=resampled_tiff.transform, # some math thing need to revise
    all_touched=True, 
    verbose=True
)

result = aggregate_to_healthsheds(
    res_poly2cell=res_poly2cell,
    raster=resampled_tiff,
    shapes=healthsheds,
    names_column="fs_uid",
    aggregation_func=agg_func,
    aggregation_name=exposure_variable
)

result.plot(column=exposure_variable, legend=True)
plt.title("{} {} (K) by Health Shed {}".format(aggregation_str, exposure_variable, input_file.stem))
plt.suptitle("Aggregation: {}, Day: {}".format(aggregation_str, str(day)))
plt.show()

UsageError: Line magic function `%%timeit` not found.


UsageError: Line magic function `%%timeit` not found.

3.2 seconds per aggregation is pretty cool!

In [ ]:
#| eval: false
result.to_parquet(here() / "data/testing/test_aggregation.parquet")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 2
      1 #| eval: false
----> 2 result.to_parquet(here() / "data/testing/test_aggregation.parquet")

NameError: name 'result' is not defined


NameError: name 'result' is not defined

For QA, we should come up with the following:

- [x] A way to list NAs in the data
- [ ] A way to visualize the data temporally
- [ ] A function to convert K to celsius

In [0]:
#| echo: false
#| output: asis
show_doc(aggregate_data)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L234){target="_blank" style="float:right; font-size:smaller"}

### aggregate_data

>      aggregate_data (cfg:omegaconf.dictconfig.DictConfig, input_file:str,
>                      output_file:str, exposure_variable:str)

*Aggregate raster data day-by-day and store all days and statistics as separate columns in a single Parquet file.*

In [ ]:
#| eval: false
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

cfg.development_mode = False
cfg.query['year'] = 2017
cfg.query['month'] = 11
cfg.query['geography'] = "nepal"

variable = "swvl1"

aggregate_data(cfg, here() / "data/input/nepal_2017_11.nc", here() / "data/testing/test_nepal_aggregation.parquet", exposure_variable=variable)

Processing daily aggregation: mean...
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 17
     13 cfg.query['geography'] = "nepal"
     15 variable = "swvl1"
---> 17 aggregate_data(cfg, here() / "data/input/nepal_2017_11.nc", here() / "data/testing/test_nepal_aggregation.parquet", exposure_variable=variable)

Cell In[1], line 36, in aggregate_data(cfg, input_file, output_file, exposure_variable)
     32 print(f"Processing daily aggregation: {daily_agg['name']}...")
     34 daily_agg_func = _get_callable(daily_agg['function'])
---> 36 with ClimateDataFileHandler(input_file) as handler:
     37     if exposure_variable in ["t2m", "d2m", "swvl1"]:
     38         ds_path = handler.get_dataset("instant")

File /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/src/era5_sandbox/core.py:242, in __enter__(self)
    240 @patch
    241 def __enter

FileNotFoundError: /net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/input/nepal_2017_11.nc does not exist

In [ ]:
#| eval: false
parquet_file = gpd.read_parquet(here() / "data/testing/test_nepal_aggregation.parquet")

---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
File ~/.conda/envs/era5_sandbox/lib/python3.11/site-packages/geopandas/io/arrow.py:653, in _read_parquet_schema_and_metadata(path, filesystem)
    652 try:
--> 653     schema = parquet.ParquetDataset(path, filesystem=filesystem, **kwargs).schema
    654 except Exception:

File ~/.conda/envs/era5_sandbox/lib/python3.11/site-packages/pyarrow/parquet/core.py:1371, in ParquetDataset.__init__(self, path_or_paths, filesystem, schema, filters, read_dictionary, memory_map, buffer_size, partitioning, ignore_prefixes, pre_buffer, coerce_int96_timestamp_unit, decryption_properties, thrift_string_size_limit, thrift_container_size_limit, page_checksum_verification, use_legacy_dataset)
   1368     partitioning = ds.HivePartitioning.discover(
   1369         infer_dictionary=True)
-> 1371 self._dataset = ds.dataset(path_or_paths, filesystem=filesystem

FileNotFoundError: [Errno 2] Failed to open local file '/net/rcstorenfs02/ifs/rc_labs/dominici_lab/lab/data_processing/csph-era5_sandbox/data/testing/test_nepal_aggregation.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
#| eval: false
parquet_file

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 2
      1 #| eval: false
----> 2 parquet_file

NameError: name 'parquet_file' is not defined


NameError: name 'parquet_file' is not defined

In [ ]:
#| eval: false
parquet_file.plot(column="day_22_daily_mean", legend=True)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
Cell In[1], line 2
      1 #| eval: false
----> 2 parquet_file.plot(column="day_22_daily_mean", legend=True)

NameError: name 'parquet_file' is not defined


NameError: name 'parquet_file' is not defined

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L321){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)